# 📰 Fake News Detection Using Text Classification
## IICT Summer Internship – Project 1

---
**Objective:** Build and compare four machine-learning classifiers (KNN, Logistic Regression, Random Forest, and MLP Neural Network) that distinguish real news from fake news using two feature representations: Bag-of-Words and TF-IDF.

**Dataset:** Kaggle Fake News Dataset – `train.csv`  
- Columns: `id`, `title`, `author`, `text`, `label` (0 = Real, 1 = Fake)  
- ~20,800 labelled articles

**Notebook Structure**

| Week | Focus |
|------|-------|
| Week 1 | Data Loading & Exploratory Data Analysis (EDA) |
| Week 2 | Text Preprocessing & Feature Engineering |
| Week 3 | Model Training (4 algorithms) |
| Week 4 | Evaluation, Visualisation & Comparative Analysis |


In [ ]:
# ── Core imports ─────────────────────────────────────────────────────────────
import os, sys, warnings
import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
from collections import Counter

# ── sklearn imports ───────────────────────────────────────────────────────────
from sklearn.model_selection     import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors           import KNeighborsClassifier
from sklearn.linear_model        import LogisticRegression
from sklearn.ensemble            import RandomForestClassifier
from sklearn.neural_network      import MLPClassifier
from sklearn.metrics             import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, auc
)
from sklearn.preprocessing       import LabelBinarizer

# ── Project modules ───────────────────────────────────────────────────────────
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))
from text_preprocessor import TextPreprocessor, preprocess_series
from feature_extractor  import ManualBoW, ManualTFIDF, SklearnTFIDF

warnings.filterwarnings('ignore')

# ── Plot style ────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor' : '#1a1a2e',
    'axes.facecolor'   : '#16213e',
    'axes.edgecolor'   : '#0f3460',
    'text.color'       : '#e0e0e0',
    'axes.labelcolor'  : '#e0e0e0',
    'xtick.color'      : '#e0e0e0',
    'ytick.color'      : '#e0e0e0',
    'axes.titlecolor'  : '#e94560',
    'axes.grid'        : True,
    'grid.color'       : '#0f3460',
    'grid.alpha'       : 0.4,
    'font.family'      : 'DejaVu Sans',
    'font.size'        : 11,
})
ACCENT = '#e94560'
BLUE   = '#0f3460'
TEAL   = '#53d8fb'
GOLD   = '#f5a623'
print('✅ All imports successful.')

---
# Week 1 — Data Loading & Exploratory Data Analysis


In [ ]:
# ── Download dataset if not already present ───────────────────────────────────
import download_data
DATA_PATH = download_data.download()
print(f'Dataset at: {DATA_PATH}')

In [ ]:
# ── Load dataset ──────────────────────────────────────────────────────────────
df = pd.read_csv(DATA_PATH)
print(f'Shape       : {df.shape}')
print(f'Columns     : {df.columns.tolist()}')
print(f'Label values: {df["label"].value_counts().to_dict()}  (0=Real, 1=Fake)')
df.head()

In [ ]:
# ── Missing-value audit ───────────────────────────────────────────────────────
print('Missing values per column:')
print(df.isnull().sum())

# Fill missing text fields with empty string
df['title']  = df['title'].fillna('')
df['author'] = df['author'].fillna('')
df['text']   = df['text'].fillna('')

# Drop rows with missing label
df.dropna(subset=['label'], inplace=True)
df['label'] = df['label'].astype(int)
print('\nAfter cleaning:')
print(df.isnull().sum())

In [ ]:
# ── EDA 1: Class Distribution ─────────────────────────────────────────────────
counts = df['label'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Class Distribution – Fake vs. Real News', fontsize=15, color=ACCENT)

# Bar chart
bars = axes[0].bar(['Real (0)', 'Fake (1)'], counts.values,
                    color=[TEAL, ACCENT], edgecolor='white', linewidth=0.6)
for bar, val in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                 f'{val:,}', ha='center', va='bottom', fontsize=11)
axes[0].set_ylabel('Count')
axes[0].set_title('Article Count by Class')

# Pie chart
axes[1].pie(counts.values, labels=['Real', 'Fake'],
            colors=[TEAL, ACCENT], autopct='%1.1f%%',
            startangle=90, textprops={'color': 'white'})
axes[1].set_title('Class Balance')

plt.tight_layout()
plt.savefig('plots/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── EDA 2: Text Length Distribution ──────────────────────────────────────────
os.makedirs('plots', exist_ok=True)
df['text_len']  = df['text'].str.split().str.len()
df['title_len'] = df['title'].str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Text Length Distribution by Class', fontsize=15, color=ACCENT)

for label_val, color, name in [(0, TEAL, 'Real'), (1, ACCENT, 'Fake')]:
    subset = df[df['label'] == label_val]['text_len'].clip(0, 2000)
    axes[0].hist(subset, bins=60, alpha=0.7, color=color, label=name)

axes[0].set_xlabel('Word Count (article body)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Article Body Length')
axes[0].legend()

for label_val, color, name in [(0, TEAL, 'Real'), (1, ACCENT, 'Fake')]:
    subset = df[df['label'] == label_val]['title_len'].clip(0, 30)
    axes[1].hist(subset, bins=30, alpha=0.7, color=color, label=name)

axes[1].set_xlabel('Word Count (title)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Title Length')
axes[1].legend()

plt.tight_layout()
plt.savefig('plots/text_length_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── EDA 3: Top-20 Most Frequent Words ────────────────────────────────────────
from text_preprocessor import TextPreprocessor
tp = TextPreprocessor()

def top_words(texts, n=20):
    counter = Counter()
    for t in texts:
        counter.update(tp.tokenize(str(t)))
    return counter.most_common(n)

real_top = top_words(df[df['label']==0]['text'])
fake_top = top_words(df[df['label']==1]['text'])

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Top 20 Words – Real vs. Fake News', fontsize=15, color=ACCENT)

for ax, data, color, title in [
    (axes[0], real_top, TEAL,  'Real News'),
    (axes[1], fake_top, ACCENT,'Fake News'),
]:
    words, freqs = zip(*data)
    ax.barh(list(reversed(words)), list(reversed(freqs)), color=color)
    ax.set_title(title)
    ax.set_xlabel('Frequency')

plt.tight_layout()
plt.savefig('plots/top_words.png', dpi=150, bbox_inches='tight')
plt.show()

---
# Week 2 — Text Preprocessing & Feature Engineering


In [ ]:
# ── Preprocess text ──────────────────────────────────────────────────────────
# Combine title + author + body for richer signal
tp = TextPreprocessor()

print('Preprocessing text (this may take ~1-2 min) …')
df['combined'] = df['title'] + ' ' + df['author'] + ' ' + df['text']
df['cleaned']  = [tp.process(t) for t in df['combined']]
print(f'Done. Sample cleaned text:\n  {df["cleaned"].iloc[0][:120]} …')

In [ ]:
# ── Train / test split ────────────────────────────────────────────────────────
X = df['cleaned'].values
y = df['label'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train samples : {len(X_train):,}')
print(f'Test  samples : {len(X_test):,}')
print(f'Train balance : {Counter(y_train)}')

In [ ]:
# ── TF-IDF feature extraction (sklearn, used by all 4 models) ─────────────────
tfidf = TfidfVectorizer(max_features=15000, ngram_range=(1,2), sublinear_tf=True)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)

print(f'TF-IDF train matrix : {X_train_tfidf.shape}')
print(f'TF-IDF test  matrix : {X_test_tfidf.shape}')

In [ ]:
# ── Manual vs. Sklearn TF-IDF comparison (first 200 docs, demo) ───────────────
DEMO_DOCS = list(X_train[:200])

manual  = ManualTFIDF(max_features=500)
sklearn_ = SklearnTFIDF(max_features=500)

X_manual_demo  = manual.fit_transform(DEMO_DOCS)
X_sklearn_demo = sklearn_.fit_transform(DEMO_DOCS)

print('Manual  TF-IDF shape :', X_manual_demo.shape)
print('Sklearn TF-IDF shape :', X_sklearn_demo.shape)
print('\nManual  – doc[0] norm :', np.linalg.norm(X_manual_demo[0]).round(4))
print('Sklearn – doc[0] norm :', np.linalg.norm(X_sklearn_demo[0]).round(4))
print('\n(Both L2-normalised → norms ≈ 1.0  ✅)')

---
# Week 3 — Model Training


In [ ]:
# ── Helper: train & evaluate one model ────────────────────────────────────────
def train_and_evaluate(name, model, X_tr, X_te, y_tr, y_te):
    print(f'\nTraining [{name}] …', end=' ', flush=True)
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    metrics = {
        'Model'    : name,
        'Accuracy' : accuracy_score(y_te, y_pred),
        'Precision': precision_score(y_te, y_pred, zero_division=0),
        'Recall'   : recall_score(y_te, y_pred, zero_division=0),
        'F1-Score' : f1_score(y_te, y_pred, zero_division=0),
    }
    print(f"Accuracy = {metrics['Accuracy']:.4f}")
    return model, y_pred, metrics

results   = []
models    = {}
all_preds = {}

In [ ]:
# ── Model 1: K-Nearest Neighbours (Non-Parametric) ───────────────────────────
knn = KNeighborsClassifier(n_neighbors=5, metric='cosine', n_jobs=-1)
m, pred, metrics = train_and_evaluate(
    'KNN (k=5)', knn,
    X_train_tfidf, X_test_tfidf, y_train, y_test
)
models['KNN']     = m
all_preds['KNN']  = pred
results.append(metrics)
print(classification_report(y_test, pred, target_names=['Real','Fake']))

In [ ]:
# ── Model 2: Logistic Regression (Parametric) ────────────────────────────────
lr = LogisticRegression(max_iter=1000, C=1.0, solver='lbfgs', n_jobs=-1)
m, pred, metrics = train_and_evaluate(
    'Logistic Regression', lr,
    X_train_tfidf, X_test_tfidf, y_train, y_test
)
models['LR']     = m
all_preds['LR']  = pred
results.append(metrics)
print(classification_report(y_test, pred, target_names=['Real','Fake']))

In [ ]:
# ── Model 3: Random Forest (Ensemble) ────────────────────────────────────────
rf = RandomForestClassifier(n_estimators=200, max_depth=None, n_jobs=-1, random_state=42)
m, pred, metrics = train_and_evaluate(
    'Random Forest', rf,
    X_train_tfidf, X_test_tfidf, y_train, y_test
)
models['RF']     = m
all_preds['RF']  = pred
results.append(metrics)
print(classification_report(y_test, pred, target_names=['Real','Fake']))

In [ ]:
# ── Model 4: MLP Neural Network (Deep Learning) ──────────────────────────────
mlp = MLPClassifier(
    hidden_layer_sizes=(256, 128, 64),
    activation='relu',
    solver='adam',
    max_iter=30,
    random_state=42,
    early_stopping=True,
    validation_fraction=0.1,
)
m, pred, metrics = train_and_evaluate(
    'MLP Neural Network', mlp,
    X_train_tfidf, X_test_tfidf, y_train, y_test
)
models['MLP']     = m
all_preds['MLP']  = pred
results.append(metrics)
print(classification_report(y_test, pred, target_names=['Real','Fake']))

---
# Week 4 — Evaluation, Visualisation & Comparative Analysis


In [ ]:
# ── Results summary table ─────────────────────────────────────────────────────
results_df = pd.DataFrame(results).set_index('Model')
results_df = results_df.round(4)
print('\n===  Model Comparison  ===')
print(results_df.to_string())
results_df

In [ ]:
# ── Bar chart: metric comparison ──────────────────────────────────────────────
metrics_cols = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
x = np.arange(len(metrics_cols))
width = 0.18
colors_ = [TEAL, ACCENT, GOLD, '#a78bfa']

fig, ax = plt.subplots(figsize=(13, 6))
for i, (model_name, color) in enumerate(zip(results_df.index, colors_)):
    vals = results_df.loc[model_name, metrics_cols].values
    bars = ax.bar(x + i*width, vals, width, label=model_name, color=color, alpha=0.85)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                f'{val:.3f}', ha='center', va='bottom', fontsize=8)

ax.set_title('Model Performance Comparison – All Metrics', fontsize=14)
ax.set_xticks(x + width*1.5)
ax.set_xticklabels(metrics_cols)
ax.set_ylim(0, 1.1)
ax.set_ylabel('Score')
ax.legend()
plt.tight_layout()
plt.savefig('plots/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Confusion matrices ────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 11))
fig.suptitle('Confusion Matrices – All Models', fontsize=15, color=ACCENT)

model_labels = [('KNN', 'KNN'), ('LR', 'Logistic Regression'),
                ('RF', 'Random Forest'), ('MLP', 'MLP Neural Network')]

for ax, (key, label) in zip(axes.flatten(), model_labels):
    cm = confusion_matrix(y_test, all_preds[key])
    sns.heatmap(cm, annot=True, fmt='d', ax=ax,
                cmap='Blues', linewidths=0.5,
                xticklabels=['Real','Fake'],
                yticklabels=['Real','Fake'])
    ax.set_title(label)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.tight_layout()
plt.savefig('plots/confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── ROC Curves ───────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 7))
ax.set_title('ROC Curves – All Models', fontsize=14)

colors_ = [TEAL, ACCENT, GOLD, '#a78bfa']
for (key, label), color in zip(model_labels, colors_):
    if hasattr(models[key], 'predict_proba'):
        proba = models[key].predict_proba(X_test_tfidf)[:, 1]
    else:
        proba = all_preds[key].astype(float)
    fpr, tpr, _ = roc_curve(y_test, proba)
    roc_auc     = auc(fpr, tpr)
    ax.plot(fpr, tpr, lw=2, color=color, label=f'{label}  (AUC = {roc_auc:.3f})')

ax.plot([0,1],[0,1], 'k--', lw=1, alpha=0.5, label='Random classifier')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.legend(loc='lower right')
ax.set_xlim([0,1])
ax.set_ylim([0,1.02])
plt.tight_layout()
plt.savefig('plots/roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Feature importance: Logistic Regression top words ────────────────────────
feature_names = tfidf.get_feature_names_out()
coefs = models['LR'].coef_[0]

top_n = 20
top_fake_idx = coefs.argsort()[-top_n:][::-1]   # highest +ve → Fake
top_real_idx = coefs.argsort()[:top_n]           # most -ve   → Real

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle('Logistic Regression — Most Informative Features', fontsize=14, color=ACCENT)

axes[0].barh([feature_names[i] for i in reversed(top_real_idx)],
              [abs(coefs[i]) for i in reversed(top_real_idx)], color=TEAL)
axes[0].set_title('Top 20 Features → Real News')
axes[0].set_xlabel('|Coefficient|')

axes[1].barh([feature_names[i] for i in reversed(top_fake_idx)],
              [coefs[i] for i in reversed(top_fake_idx)], color=ACCENT)
axes[1].set_title('Top 20 Features → Fake News')
axes[1].set_xlabel('Coefficient')

plt.tight_layout()
plt.savefig('plots/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Parametric vs. Non-Parametric Analysis ────────────────────────────────────
print('=' * 65)
print('  Parametric vs. Non-Parametric Model Discussion')
print('=' * 65)
print('''
PARAMETRIC models (Logistic Regression, MLP Neural Network)
  • Make explicit assumptions about the data distribution.
  • Have a fixed, small set of learnable parameters.
  • Generally faster at inference once trained.
  • Logistic Regression is interpretable (linear decision boundary).
  • MLP can learn non-linear patterns but is a "black box".

NON-PARAMETRIC models (KNN, Random Forest)
  • Make no explicit distributional assumptions.
  • KNN: stores the entire training set; classifies by majority vote
         among k nearest neighbours in TF-IDF space.
  • Random Forest: ensemble of decision trees → robust to outliers.
  • Both tend to generalise well on noisy text data.
''')

best = results_df['F1-Score'].idxmax()
print(f'  Best overall model by F1-Score: {best}')
print(f'  F1 = {results_df.loc[best, "F1-Score"]:.4f}')

In [ ]:
# ── Save best model ───────────────────────────────────────────────────────────
import pickle, os
os.makedirs('models', exist_ok=True)

best_key = {'KNN':'KNN','Logistic Regression':'LR','Random Forest':'RF','MLP Neural Network':'MLP'}[best]

with open('models/best_model.pkl', 'wb') as f:
    pickle.dump({'vectorizer': tfidf, 'model': models[best_key], 'name': best}, f)

print(f'Best model saved → models/best_model.pkl  (model: {best})')
print('\n🏁 Project 1 complete!')